In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 
import glob
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib import colors
import matplotlib.ticker as ticker

In [2]:
# Read in data
cells = pd.read_csv('../data/myeloid_panel_umap.csv', index_col=0)
cells.head()

,209Bi_CD45,Center,161Dy_CD274_PDL1,162Dy_CD80,Event_length,157Gd,113In_CD45,191Ir_DNA1,193Ir_DNA2,175Lu_STAT1_PE,...,173Yb_CD47_Biotin_asinh_coarseAlign_fineAlign,174Yb_HLA-DR_asinh_coarseAlign_fineAlign,176Yb_CD56_asinh_coarseAlign_fineAlign,Alignment_MC_fineAlign,FlowSOM_cluster,FlowSOM_metacluster,FlowSOM_cluster.1,FlowSOM_metacluster.1,UMAP_X,UMAP_Y
1,0.00000,0.000,3.88961,0.0,22,0.00000,0.0,158.923,307.365,16.45780,...,4.771649,1.059669,0.556633,2,143,24,6,2,-3.129287,6.578265
2,0.00000,949.553,2.01143,0.0,20,0.00000,0.0,287.998,537.970,27.61740,...,4.522643,2.077372,0.136398,6,27,12,131,10,-4.816134,-4.346446
3,1.17233,723.309,2.35961,0.0,19,0.00000,0.0,169.809,299.578,5.22007,...,5.787666,0.783705,0.062180,7,19,9,88,9,-0.664257,-5.016717
4,0.00000,868.160,1.08257,0.0,21,2.85382,0.0,188.745,349.081,13.24680,...,5.552658,2.958660,0.136398,6,41,12,103,10,-4.368270,-4.707629
5,4.20128,0.000,0.00000,0.0,17,0.00000,0.0,109.358,222.585,3.00359,...,5.424986,0.603684,0.820364,7,19,9,88,9,-0.691456,-4.948173


In [3]:
# Create alias dictionaries
alias = pd.read_excel('../data/patient_alias_from_2025.xlsx')
alias_dict = dict(zip(alias['PID'].astype(str), alias['alias']))

control = pd.read_csv('../data/healthy_control.csv')
control_dict = dict(zip(control['key'].str.replace('_T_Cell_Panel.fcs', ''), control['value'].str.replace('.fcs', '', regex=False)))
control_dict = {k.replace('Healthy_BMA_67y_Male', 'Healthy_BMA_67Y_Male'): v for k, v in control_dict.items()}

<ipython-input-3-1ab1d6a91885>:6: FutureWarning: The default value of regex will change from True to False in a future version.
  control_dict = dict(zip(control['key'].str.replace('_T_Cell_Panel.fcs', ''), control['value'].str.replace('.fcs', '', regex=False)))


In [4]:
# Replace the patient identifiers in the file with our agreed alias
pid_list = []
for i in cells['Patient_ID']:
    if i.startswith('6'):
        x = alias_dict[i.replace('_', '')]
        pid_list.append(x)
    elif i.startswith('He'):
        x = control_dict[i]
        pid_list.append(x)
        
cells['Patient_ID'] = pid_list    

In [5]:
# Replace the patient identifiers in the FileName column with our agreed alias
file_list = []
for i in cells['FileName']:
    if i.startswith('6'):
        x = i.split('_')[0] + i.split('_')[1] 
        x = alias_dict[x] + '_'
        y = '_'.join(i.split('_')[2:])
        z = x + y
        file_list.append(z)
    elif i.startswith('Healthy'):
        x = i.replace('_Myeloid_Panel', '')
        x = control_dict[x] + '_Myeloid_Panel'
        file_list.append(x)
        
cells['FileName'] = file_list

In [6]:
cells['7_month_response'].unique()

array(['Non_responder', 'Responder', 'Unsure', 'Healthy'], dtype=object)

In [7]:
# maybe remove 'Batch_Control', 'Reference', 'Batch_Control_Type'
# cells['Batch_Control']
# update the '7_month_response', '12_month_response'

In [8]:
cells.columns

Index(['209Bi_CD45', 'Center', '161Dy_CD274_PDL1', '162Dy_CD80',
       'Event_length', '157Gd', '113In_CD45', '191Ir_DNA1', '193Ir_DNA2',
       '175Lu_STAT1_PE', '104Pd_CD45', '106Pd_CD45', '108Pd_CD45',
       '110Pd_CD45', '141Pr_CD196_CCR6', '194Pt_Cisplatin1',
       '195Pt_Cisplatin2', '147Sm_NKG2DFc_Cy5', '159Tb_CD90', '89Y_CD45',
       'Time', 'FileName', 'FileNo', 'V1', 'Date', 'Patient_ID', 'Timepoint',
       'Batch_Control', 'Reference', 'Batch_Control_Type', 'Healthy',
       'Disease', 'Batch', '7_month_response', '12_month_response',
       '163Dy_CD117_asinh_coarseAlign_fineAlign',
       '164Dy_CD172ab_SIRPab_asinh_coarseAlign_fineAlign',
       '166Er_CD34_asinh_coarseAlign_fineAlign',
       '167Er_CD38_asinh_coarseAlign_fineAlign',
       '168Er_Ki67_asinh_coarseAlign_fineAlign',
       '170Er_CD3_asinh_coarseAlign_fineAlign',
       '151Eu_CD123_asinh_coarseAlign_fineAlign',
       '153Eu_CD68_asinh_coarseAlign_fineAlign',
       '155Gd_CD303_asinh_coarseAlign_fi

In [9]:
control_dict

{'Healthy_BMA_51Y_Male': 'Control_1',
 'Healthy_BMA_from_Shiv_2': 'Control_2',
 'Healthy_BMA_from_Shiv_1': 'Control_3',
 'Healthy_BMA_HSA4496': 'Control_4',
 'Healthy_BMA_HSA1576': 'Control_5',
 'Healthy_BMA_HSA1385': 'Control_6',
 'Healthy_BMA_83Y_Male': 'Control_7',
 'Healthy_BMA_HSA1531': 'Control_8',
 'Healthy_BMA_80Y_Male': 'Control_9',
 'Healthy_BMA_HSA4362': 'Control_10',
 'Healthy_BMA_HSA4848': 'Control_11',
 'Healthy_BMA_HSA4696': 'Control_12',
 'Healthy_BMA_67Y_Male': 'Control_13',
 'Healthy_BMA_51Y_Female': 'Control_14',
 'Healthy_BMA_84Y_Female': 'Control_15',
 'Healthy_BMA_HSA2620': 'Control_16',
 'Healthy_BMA_61Y_Male': 'Control_17',
 'Healthy_BMA_73Y_Female': 'Control_18',
 'Healthy_BMA_63Y_Female': 'Control_19',
 '036-04-69': 'Control_20',
 '1115-50-04': 'Control_21',
 '926-23-04': 'Control_22',
 'HSA1620': 'Control_23',
 'HSA2146': 'Control_24'}

In [11]:
cells.to_csv('../data/myeloid_panel_umap_patient_updated.csv')